# Лабораторная 3: GRU encoder-decoder для задачи seq2seq 

## Цель
Реализовать модель `seq2seq` на базе `GRU`, которая разворачивает входную последовательность в обратном порядке.

Формат `seq2seq` означает: длина входа и длина выхода могут различаться, а генерация выхода идет пошагово.

## Контракт данных
Используются целочисленные токены:
- `1..9` — содержательные значения;
- `PAD=0` — заполнение;
- `SOS=10` — старт декодера;
- `EOS=11` — конец последовательности.

Формируются три связанных тензора:
- `encoder_input`;
- `decoder_input`;
- `decoder_target`.


## Таблица форм тензоров

| Тензор | Форма | Смысл | Где используется |
|---|---|---|---|
| `encoder_input` | `(N, T_in)` | Вход кодировщика | `model.fit`/`model.predict` |
| `decoder_input` | `(N, T_out)` | Вход декодера со сдвигом | `model.fit`/`model.predict` |
| `decoder_target` | `(N, T_out, 1)` | Истинные выходные токены | Функция потерь |
| `probs` | `(N_test, T_out, V)` | Распределения вероятностей по словарю | `model.predict` |
| `preds` | `(N_test, T_out)` | Предсказанные индексы токенов | Оценка |
| `mask` | `(N_test, T_out)` | Маска значимых токенов (`!= PAD`) | `exact_match` |
| `exact_match` | скаляр | Доля полностью верных последовательностей | Итоговая метрика |


## Контракт модели
- В `model.fit` передается список входов `[encoder_input, decoder_input]` и `decoder_target`.
- Выход модели: распределение вероятностей по словарю на каждом шаге декодера.
- Для функции потерь `sparse_categorical_crossentropy` целевые значения задаются целыми индексами классов.


## Мини-теория
`seq2seq` с кодировщиком и декодером в нотации теории:

$$
h_t^{enc} = f_{enc}(x_t, h_{t-1}^{enc}), \quad c_{enc}=h_{T_{in}}^{enc}
$$

$$
h_t^{dec} = f_{dec}(E(y_{t-1}^{in}), h_{t-1}^{dec}, c_{enc}), \quad
s_t^{dec} = W_{out}h_t^{dec} + b_{out}, \quad
\hat{y}_t = \mathrm{softmax}(s_t^{dec})
$$

На обучении используется обучение с учителем по предыдущему токену (`teacher forcing`).


In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split


def set_seed(seed: int = 42) -> None:
    np.random.seed(seed)
    tf.random.set_seed(seed)


set_seed(42)
print('Версия TensorFlow:', tf.__version__)

## Генерация данных
**Что сделать:** подготовить `encoder_input`, `decoder_input`, `decoder_target` со сдвигом на один шаг.

**Почему:** декодер обучается предсказывать следующий токен по предыдущему.

**Ожидаемые формы:** `(N,T_in)`, `(N,T_out)`, `(N,T_out,1)`.

**Частая ошибка:** неверный сдвиг между входом и целями декодера.


In [ ]:
PAD_ID = 0
SOS_ID = 10
EOS_ID = 11
VOCAB_SIZE = 12
ENC_LEN = 6
DEC_LEN = ENC_LEN + 1


def make_one_sample(min_len: int = 3, max_len: int = ENC_LEN):
    length = np.random.randint(min_len, max_len + 1)
    seq = np.random.randint(1, 10, size=length, dtype=np.int32)
    # TODO 1: получите обратную последовательность
    rev = ...
    return seq, rev


def pad_sequence(seq, target_len, pad_value=PAD_ID):
    out = np.full((target_len,), pad_value, dtype=np.int32)
    out[:len(seq)] = seq
    return out


def make_dataset(n_samples: int = 7000):
    encoder_input = np.zeros((n_samples, ENC_LEN), dtype=np.int32)
    decoder_input = np.zeros((n_samples, DEC_LEN), dtype=np.int32)
    decoder_target = np.zeros((n_samples, DEC_LEN), dtype=np.int32)

    for i in range(n_samples):
        seq, rev = make_one_sample()
        enc = pad_sequence(seq, ENC_LEN)

        # TODO 2: сформируйте вход декодера: [SOS] + rev
        dec_in = ...
        # TODO 3: сформируйте целевой выход декодера: rev + [EOS]
        dec_out = ...

        encoder_input[i] = pad_sequence(enc, ENC_LEN)
        decoder_input[i] = pad_sequence(dec_in, DEC_LEN)
        decoder_target[i] = pad_sequence(dec_out, DEC_LEN)

    return encoder_input, decoder_input, decoder_target[..., None]


enc_in, dec_in, dec_tgt = make_dataset()

enc_train, enc_test, dec_in_train, dec_in_test, dec_tgt_train, dec_tgt_test = train_test_split(
    enc_in,
    dec_in,
    dec_tgt,
    test_size=0.2,
    random_state=42,
)

print('Форма enc_train   :', enc_train.shape)
print('Форма dec_in_train:', dec_in_train.shape)
print('Форма dec_tgt_train:', dec_tgt_train.shape)

In [ ]:
# Мини-проверка данных
assert enc_in.ndim == 2 and dec_in.ndim == 2, 'encoder_input и decoder_input должны быть двумерными'
assert dec_tgt.ndim == 3 and dec_tgt.shape[-1] == 1, 'decoder_target должен иметь форму (N,T,1)'
assert enc_in.shape[1] == ENC_LEN and dec_in.shape[1] == DEC_LEN
print('Мини-проверка данных: OK')

## Модель
Используется гибридная схема:
- кодировщик и встраивание декодера собираются через `Sequential.add(...)`;
- объединение двух входов выполняется через функциональный граф (`Functional API`).

Такой вариант сохраняет прозрачную структуру и поддерживает два входа `seq2seq`.


In [ ]:
def build_model(vocab_size: int = VOCAB_SIZE, emb_dim: int = 24, latent_dim: int = 48):
    encoder_inputs = tf.keras.layers.Input(shape=(ENC_LEN,), name='encoder_inputs')

    encoder_seq = tf.keras.Sequential(name='encoder_seq')
    # TODO 4: добавьте слой Embedding в encoder_seq
    encoder_seq.add(...)
    # TODO 5: добавьте слой GRU в encoder_seq для получения контекста
    encoder_seq.add(...)
    context = encoder_seq(encoder_inputs)

    decoder_inputs = tf.keras.layers.Input(shape=(DEC_LEN,), name='decoder_inputs')
    decoder_seq = tf.keras.Sequential(name='decoder_seq')
    # TODO 6: добавьте слой Embedding в decoder_seq
    decoder_seq.add(...)
    dec_emb = decoder_seq(decoder_inputs)

    # TODO 7: создайте GRU декодера (return_sequences=True, initial_state=context)
    dec_gru = ...
    dec_out = ...

    head_seq = tf.keras.Sequential(name='output_head')
    # TODO 8: добавьте Dense(VOCAB_SIZE, activation='softmax')
    head_seq.add(...)
    outputs = head_seq(dec_out)

    model = tf.keras.Model([encoder_inputs, decoder_inputs], outputs, name='gru_seq2seq')
    # TODO 9: скомпилируйте модель (adam, sparse_categorical_crossentropy, accuracy)
    model.compile(...)
    return model


model = build_model()
model.summary()

In [ ]:
# Мини-проверка модели
assert model.output_shape[1] == DEC_LEN, 'Длина выхода должна совпадать с DEC_LEN'
assert model.output_shape[2] == VOCAB_SIZE, 'Последняя ось должна быть равна размеру словаря'
print('Мини-проверка модели: OK')

## Трассировка одного примера через модель
Проверка форм тензоров:
1. входы кодировщика и декодера;
2. выход рекуррентного слоя декодера;
3. итоговое распределение вероятностей по словарю.


In [ ]:
sample_enc = enc_train[:1]
sample_dec = dec_in_train[:1]

trace_model = tf.keras.Model(
    inputs=model.inputs,
    outputs=[model.get_layer('dec_gru').output, model.output],
)

dec_gru_out, probs_out = trace_model.predict([sample_enc, sample_dec], verbose=0)

print('Вход encoder_input :', sample_enc.shape)
print('Вход decoder_input :', sample_dec.shape)
print('После GRU декодера :', dec_gru_out.shape)
print('После выходного слоя:', probs_out.shape)
print('Первые 3 предсказанных индекса:', probs_out.argmax(axis=-1)[0, :3])

## Как идет обучение внутри эпохи
По каждому мини-батчу выполняется цикл:
1. прямой проход (`forward pass`) для `encoder` и `decoder`;
2. расчет функции потерь по всем шагам декодера;
3. обратное распространение через время (`BPTT`) через оба рекуррентных блока;
4. обновление параметров.

Для `GRU` управление памятью задается воротами обновления и сброса, что обычно дает компромисс между устойчивостью обучения и скоростью по сравнению с более тяжелой `LSTM`.


## Обучение
В `fit` подается пара входов `[encoder_input, decoder_input]` и целевой тензор `decoder_target`.

Для контроля обобщения используется `validation_split=0.2` на обучающей части.


In [ ]:
def train_model(model, enc_train, dec_in_train, dec_tgt_train, epochs: int = 16):
    # TODO 10: обучите модель с validation_split=0.2 и batch_size=64
    history = model.fit(...)
    return history


history = train_model(model, enc_train, dec_in_train, dec_tgt_train)

In [ ]:
# Мини-проверка обучения
assert 'val_loss' in history.history and 'val_accuracy' in history.history
print('Последняя val_accuracy:', float(history.history['val_accuracy'][-1]))
print('Мини-проверка обучения: OK')

## Оценка и диагностика
Считаются:
- `token_accuracy` — доля корректных токенов;
- `exact_match` — доля последовательностей, полностью совпавших по значимым токенам.

`exact_match` вычисляется с маской `target != PAD`.


In [ ]:
def evaluate_model(model, enc_test, dec_in_test, dec_tgt_test):
    loss, token_acc = model.evaluate([enc_test, dec_in_test], dec_tgt_test, verbose=0)
    probs = model.predict([enc_test, dec_in_test], verbose=0)
    preds = probs.argmax(axis=-1)
    target = dec_tgt_test[:, :, 0]
    mask = (target != PAD_ID)

    # TODO 11: вычислите exact_match по значимым токенам
    exact_match = ...

    return {
        'loss': float(loss),
        'token_accuracy': float(token_acc),
        'exact_match': float(exact_match),
        'preds': preds,
        'target': target,
        'mask': mask,
        'probs': probs,
    }


metrics = evaluate_model(model, enc_test, dec_in_test, dec_tgt_test)
print({k: v for k, v in metrics.items() if k in ('loss', 'token_accuracy', 'exact_match')})

In [ ]:
# Мини-проверка метрик
assert 0.0 <= metrics['token_accuracy'] <= 1.0
assert 0.0 <= metrics['exact_match'] <= 1.0
if metrics['exact_match'] >= 0.80:
    print('Целевой порог достигнут: exact_match >= 0.80')
else:
    print('Целевой порог не достигнут: проверьте сдвиг decoder_input/decoder_target и параметры обучения')

## Что ожидать на практике
- В начале обучения `token_accuracy` может расти быстрее `exact_match`.
- Малейшая ошибка в середине последовательности резко снижает `exact_match`.
- Если обе метрики не растут, сначала проверить корректность сдвига `decoder_input` и `decoder_target`.
- Для корректной реализации обычно достигается `exact_match >= 0.80`.


In [ ]:
# Визуализация динамики обучения
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.title('Динамика функции потерь')
plt.xlabel('Эпоха')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='train_token_accuracy')
plt.plot(history.history['val_accuracy'], label='val_token_accuracy')
plt.title('Динамика token_accuracy')
plt.xlabel('Эпоха')
plt.legend()

plt.tight_layout()
plt.show()

## Вопросы для самопроверки
1. Почему для `seq2seq` требуется два входа в `model.fit`?
2. Как сдвиг `decoder_input`/`decoder_target` связан с `teacher forcing`?
3. Почему `exact_match` обычно ниже `token_accuracy`?
4. Как маска `target != PAD` влияет на итоговую оценку качества?


## Типичные ошибки (симптом -> причина -> исправление)
- Низкий `exact_match` при приемлемом `token_accuracy` -> нарушен сдвиг токенов -> проверить `[SOS] + rev` и `rev + [EOS]`.
- Ошибка функции потерь -> неверная форма `decoder_target` -> использовать форму `(N,T,1)` с целыми индексами.
- Нестабильная метрика последовательности -> PAD учитывается как обычный токен -> считать метрику по маске значимых позиций.
- Обучение не запускается -> передан только один вход -> передавать `[encoder_input, decoder_input]`.
